# 02 – Preprocessing
## HumanForYou – Attrition ML

## SECTION – IMPORT + LOAD

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

general = pd.read_csv(os.path.join(RAW_DIR, 'general_data.csv'))
employee_survey = pd.read_csv(os.path.join(RAW_DIR, 'employee_survey_data.csv'))
manager_survey = pd.read_csv(os.path.join(RAW_DIR, 'manager_survey_data.csv'))

print(f'general: {general.shape}')
print(f'employee_survey: {employee_survey.shape}')
print(f'manager_survey: {manager_survey.shape}')

general: (4410, 24)
employee_survey: (4410, 4)
manager_survey: (4410, 3)


## SECTION – MERGE DATASETS

In [2]:
df_merged = general.merge(employee_survey, on='EmployeeID', how='inner') \
                    .merge(manager_survey, on='EmployeeID', how='inner')

print(f'Shape apres merge: {df_merged.shape}')
assert df_merged.shape[0] == 4410

Shape apres merge: (4410, 29)


## SECTION – SUPPRESSION VARIABLES NON INFORMATIVES

In [3]:
for col in ['EmployeeCount', 'Over18', 'StandardHours']:
    print(f'{col}: {df_merged[col].nunique()} valeur(s) unique(s) -> {df_merged[col].unique()}')

df_cleaned = df_merged.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])
print(f'\nShape apres suppression: {df_cleaned.shape}')

EmployeeCount: 1 valeur(s) unique(s) -> [1]
Over18: 1 valeur(s) unique(s) -> <StringArray>
['Y']
Length: 1, dtype: str
StandardHours: 1 valeur(s) unique(s) -> [8]

Shape apres suppression: (4410, 26)


## SECTION – TRAITEMENT VALEURS MANQUANTES

In [4]:
print('Valeurs manquantes avant imputation:')
missing_before = df_cleaned.isnull().sum()
print(missing_before[missing_before > 0])

for col in df_cleaned.columns:
    if df_cleaned[col].isnull().sum() > 0:
        if df_cleaned[col].dtype in ['float64', 'int64']:
            median_val = df_cleaned[col].median()
            df_cleaned[col] = df_cleaned[col].fillna(median_val)
            print(f'  {col} -> imputation mediane ({median_val})')
        else:
            mode_val = df_cleaned[col].mode()[0]
            df_cleaned[col] = df_cleaned[col].fillna(mode_val)
            print(f'  {col} -> imputation mode ({mode_val})')

print(f'\nValeurs manquantes apres imputation: {df_cleaned.isnull().sum().sum()}')

Valeurs manquantes avant imputation:
NumCompaniesWorked         19
TotalWorkingYears           9
EnvironmentSatisfaction    25
JobSatisfaction            20
WorkLifeBalance            38
dtype: int64
  NumCompaniesWorked -> imputation mediane (2.0)
  TotalWorkingYears -> imputation mediane (10.0)
  EnvironmentSatisfaction -> imputation mediane (3.0)
  JobSatisfaction -> imputation mediane (3.0)
  WorkLifeBalance -> imputation mediane (3.0)

Valeurs manquantes apres imputation: 0


## SECTION – ENCODAGE CATEGORIEL

In [5]:
df_cleaned['Attrition'] = df_cleaned['Attrition'].map({'Yes': 1, 'No': 0})

cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']
print(f'Colonnes categorielles a encoder: {cat_cols}')
for col in cat_cols:
    print(f'  {col}: {df_cleaned[col].nunique()} modalites')

df_final = pd.get_dummies(df_cleaned, columns=cat_cols, drop_first=True, dtype=int)
print(f'\nShape apres encodage: {df_final.shape}')

Colonnes categorielles a encoder: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']
  BusinessTravel: 3 modalites
  Department: 3 modalites
  EducationField: 6 modalites
  Gender: 2 modalites
  JobRole: 9 modalites
  MaritalStatus: 3 modalites

Shape apres encodage: (4410, 40)


## SECTION – EXPORT + VALIDATION

In [6]:
assert df_final.shape == (4410, 40), f'Shape inattendu: {df_final.shape}'
assert df_final.isna().sum().sum() == 0, 'Il reste des NaN'
assert 'EmployeeID' in df_final.columns, 'EmployeeID manquant'

os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path = os.path.join(PROCESSED_DIR, 'cleaned_attrition_base.csv')
df_final.to_csv(output_path, index=False)

print(f'df_final.shape: {df_final.shape}')
print(f'NaN total: {df_final.isna().sum().sum()}')
print(f'EmployeeID present: {"EmployeeID" in df_final.columns}')
print(f'Export OK : {output_path}')

df_final.shape: (4410, 40)
NaN total: 0
EmployeeID present: True
Export OK : ..\data\processed\cleaned_attrition_base.csv
